# Rotational interaction

Mode that creates ghost atoms following interactions and guiding interacted atoms to match rotation of the controller:

<video controls src="./assets/rotational_interaction.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation
from nanover.trajectory import FrameData
from nanover.mdanalysis import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("trypsin_benzamidine.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: rotational interaction")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities, SceneObjectsUtility

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_interaction_modes()
utilities.use_transform_handles()

In [3]:
structure_atoms = universe.select_atoms("not resname BEN")
molecule_atoms = universe.select_atoms("resname BEN and not name H*")

utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=universe.select_atoms("resname BEN").atoms.indices)

## Ghosts

In [4]:
import numpy as np
import MDAnalysis as mda
from nanover.utilities.transforms import Transform


def make_ghost(key, atoms, positions=None):
    # extract ghost molecule
    ghost_universe = mda.Merge(atoms)
    ghost_positions = positions if positions is not None else ghost_universe.atoms.positions / 10  # angstrom -> nm
    ghost_bond_pairs = ghost_universe.bonds.indices

    # normalise around centroid, determine bounding radius
    centroid = np.mean(ghost_positions, axis=0)
    ghost_positions -= centroid

    # transform + handle for manipulating it
    utilities.transforms.update_transform(f"ghost.{key}", transform=Transform.from_translation(centroid), parent="simulation")

    visuals = SceneObjectsUtility.from_runner(imd_runner)

    # add transparent spheres and lines to scene at positions relative to nanotube in first frame:
    for i, position in enumerate(ghost_positions):
        visuals.update_shape(f"ghost.{key}.{i}", position=position, size=0.1, color=[1.0, 1.0, 1.0, 0.5], parent=f"ghost.{key}")
    for i, (a, b) in enumerate(ghost_bond_pairs):
        visuals.update_line(f"ghost.{key}.{i}", positions=ghost_positions[[a, b]], size=0.05, color=[1.0, 1.0, 1.0, 0.5], parent=f"ghost.{key}")

    return f"ghost.{key}", ghost_positions, visuals

In [5]:
from dataclasses import dataclass

import numpy.typing as npt

from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from nanover.jupyter.transform_grabbing import TransformGrabbingContext

def get_cursor_id_from_interaction(interaction: ParticleInteraction):
    owner_id = interaction.properties.get("owner.id", "")
    hand = interaction.properties.get("label", "hand.").removeprefix("hand.")
    return f"cursor.{owner_id}.{hand}"


@dataclass(kw_only=True)
class GrabData:
    indices: list[int]
    positions: npt.NDArray
    visuals: SceneObjectsUtility
    follower: GhostFollowerAgent


grabbing = TransformGrabbingContext[GrabData].from_utilities(utilities)


class RotationalInteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        # try to associate interaction with cursor
        cursor_id = get_cursor_id_from_interaction(interaction)
        cursor = utilities.get_shared_state_value(cursor_id)

        # if there's no associated cursor, ignore
        if cursor is None:
            return

        # make a ghost of the interacted atoms using current particle positions
        positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions[interaction.particles]
        ghost_id, ghost_positions, visuals = make_ghost(cursor_id, universe.atoms[interaction.particles], positions)

        # grab the ghost transform with the cursor
        grab = grabbing.start_grab_from_cursor(cursor_id, transform_id=ghost_id, cursor=cursor, scale=False)
        assert grab is not None

        # start a ghost follower agent for this interaction
        follower = GhostFollowerAgent.from_runner(imd_runner)

        # attach extra data to the grab for later cleanup
        grab.data = GrabData(
            positions=ghost_positions,
            indices=[int(i) for i in interaction.particles],
            visuals=visuals,
            follower=follower,
        )

        follower.grab = grab
        follower.cursor_id = cursor_id
        follower.visuals = visuals
        follower.start()

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        # try to associate interaction with cursor
        cursor_id = get_cursor_id_from_interaction(interaction)
        grab = grabbing.end_grab(cursor_id)

        # if the associated cursor was grabbing, do grab cleanup (stop follower, clear ghost visuals)
        if grab is not None and grab.data is not None:
            grab.data.visuals.clear()
            grab.data.follower.close()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        grabbing.update_grab_from_cursor(key, cursor=cursor)

utilities.add_interaction_mode(RotationalInteractMode, "rotational interaction", icon="🔃")

## Ghost follower

In [6]:
from nanover.jupyter import ImdAgent
from nanover.jupyter.utilities import SceneObjectsUtility
from nanover.jupyter.transform_grabbing import TransformGrabData
from nanover.imd import ParticleInteraction


class GhostFollowerAgent(ImdAgent):
    cursor_id: str
    grab: TransformGrabData[GrabData]
    visuals: SceneObjectsUtility

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        grab = self.grab
        cursor_id = self.cursor_id

        # target positions are original ghost positions transformed by ghost transform
        target_positions = utilities.transforms.fetch_transform(grab.transform_id).points_local_to_parent(grab.data.positions)
        real_positions = full_frame.particle_positions[grab.data.indices]

        target_centroid = target_positions.mean(axis=0)
        real_centroid = real_positions.mean(axis=0)

        self.visuals.update_line(f"{cursor_id}.follow.centroid", positions=[real_centroid, target_centroid], size=0.01, color=[1.0, 0, 0, 1.0])
        self.interactions.update_interaction(f"{cursor_id}.follow.centroid", ParticleInteraction(
            position=target_centroid,
            particles=[int(x) for x in grab.data.indices],
            type="spring",
            scale=500,
            max_force=100,
        ))

        # rotational following if centroid is close enough
        close = np.linalg.norm(real_centroid - target_centroid, axis=0) < 1

        # find target positions ignoring centroid differences
        rotational_target = target_positions - target_centroid
        rotational_real = real_positions - real_centroid
        rotational = real_positions + (rotational_target - rotational_real)

        for i, index in enumerate(grab.data.indices):
            if close:
                self.visuals.update_line(f"{cursor_id}.follow.{i}", positions=[rotational[i], real_positions[i]], size=0.01, color=[1.0, 0, 0, 1.0])
                self.interactions.update_interaction(f"{cursor_id}.follow.{i}", ParticleInteraction(
                    position=rotational[i],
                    particles=[int(index)],
                    type="spring",
                    scale=100,
                    max_force=50,
                ))
            else:
                self.visuals.remove_line(f"{cursor_id}.follow.{i}")
                self.interactions.remove_interaction(f"{cursor_id}.follow.{i}")